In [1]:
import os
import torch
import pandas as pd
import numpy as np
import pickle
from eval import season_performance_with_unlimited_transfers, expected_fpl_points
import json
from model import FPLComponentModel

In [2]:
base_path = os.getcwd()
base_path

'/Users/bragehs/Documents/FPL_forecast/predictor'

In [3]:
data_path = os.path.join(base_path, 'processed_data')
data_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [4]:
X_test_numeric = torch.load(data_path + '/X_test.pt', weights_only=True)
X_test_static = torch.load(data_path + '/X_static_test.pt', weights_only=True)
y_test = torch.load(data_path + '/y_test.pt', weights_only=True)
test_mapping = pd.read_csv(data_path + '/test_mapping.csv')

In [5]:
best_model_data = torch.load("best_model_xg.pth", map_location=torch.device('cpu'))

/var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/ipykernel_71832/1870094898.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_model_data = torch.load("best_model_xg.p

In [6]:
print(best_model_data.keys())
for k, v in best_model_data['model_state_dict'].items():
    if 'embedding' in k:
        print(k, v.shape)

dict_keys(['model_state_dict', 'best_performance', 'hidden_dim', 'lstm_layers', 'dropout'])


In [7]:
print(best_model_data['hidden_dim'])
print(best_model_data['lstm_layers'])

192
2


In [8]:
model = FPLComponentModel(
            numeric_seq_dim=X_test_numeric.shape[-1],
            static_dim=X_test_static.shape[-1],
            hidden_dim=best_model_data['hidden_dim'],
            lstm_layers=best_model_data["lstm_layers"],
            dropout=0.0,
        )
model.load_state_dict(best_model_data['model_state_dict'])

<All keys matched successfully>

In [9]:
#print number of parameters in the model
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of parameters in the model: {num_params}")

Number of parameters in the model: 588230


In [10]:
model.eval()

FPLComponentModel(
  (locked_dropout_in): LockedDropout()
  (lstm): LSTM(19, 192, num_layers=2, batch_first=True)
  (locked_dropout_out): LockedDropout()
  (attn_pool): AttentionPool(
    (proj): Sequential(
      (0): Linear(in_features=192, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=1, bias=True)
    )
  )
  (out_dropout): Dropout(p=0.0, inplace=False)
  (backbone): Sequential(
    (0): Linear(in_features=597, out_features=192, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
  (head): Linear(in_features=192, out_features=5, bias=True)
)

In [11]:
test = pd.read_csv(data_path + '/test_data.csv')

In [12]:
test.columns

Index(['GW', 'last_1_assists', 'last_1_bonus', 'last_1_creativity',
       'last_1_clean_sheets', 'last_1_goals_conceded', 'last_1_goals_scored',
       'last_1_ict_index', 'last_1_influence', 'last_1_minutes',
       'last_1_threat', 'last_1_red_cards', 'last_1_yellow_cards',
       'last_1_team_score', 'last_1_opponent_score', 'last_1_expected_goals',
       'last_1_expected_assists', 'last_1_expected_goals_conceded',
       'last_all_assists', 'last_all_bonus', 'last_all_creativity',
       'last_all_clean_sheets', 'last_all_goals_conceded',
       'last_all_goals_scored', 'last_all_ict_index', 'last_all_influence',
       'last_all_minutes', 'last_all_threat', 'last_all_red_cards',
       'last_all_yellow_cards', 'last_all_team_score',
       'last_all_opponent_score', 'last_all_expected_goals',
       'last_all_expected_assists', 'last_all_expected_goals_conceded',
       'fixture_difficulty', 'position_encoded_1.0', 'position_encoded_2.0',
       'position_encoded_3.0', 'position

In [13]:
test["minutes"].describe()

count    27283.00000
mean        27.43397
std         38.04890
min          0.00000
25%          0.00000
50%          0.00000
75%         72.00000
max         90.00000
Name: minutes, dtype: float64

In [14]:
output = model(X_test_numeric, X_test_static)
positions = X_test_static[:, -4:]  # last 4 static features are one-hot position
points_pred = expected_fpl_points(output, positions).detach().numpy()  
y_test = y_test.squeeze(1)

In [15]:
print(points_pred.shape)
print(y_test.shape)

(27283,)
torch.Size([27283])


In [16]:
print(torch.mean(y_test))
print(torch.var(y_test))    

tensor(1.1469)
tensor(5.3394)


In [17]:
print(np.mean(points_pred))
print(np.var(points_pred))
print(np.max(points_pred))

1.0482885
1.8077415
6.9256983


In [18]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

rmse = root_mean_squared_error(y_test, points_pred)
print(f"RMSE: {rmse}")

mae = mean_absolute_error(y_test, points_pred)
print(f"MAE: {mae}")


RMSE: 1.9538114070892334
MAE: 0.9686686992645264


In [19]:
X_test_numeric.shape

torch.Size([27283, 5, 19])

In [20]:
scores, total_score = season_performance_with_unlimited_transfers(
    y_test=y_test,
    predictions=points_pred,
)

Players with NaN total_points_last_season: []
Number of NaN values remaining: 0
1.0 :  lukasz_fabianski
2.0 :  sepp_van_den_berg
3.0 :  tyler_dibling
4.0 :  daniel_jebbison
Bench players: ['lukasz_fabianski', 'sepp_van_den_berg', 'tyler_dibling', 'daniel_jebbison']
Bench cost: 170.0
Simulating season with unlimited transfers for 38 gameweeks
Available budget per gameweek: 830.0

--- Gameweek 1.0 ---
Players available for GW 1.0: 668
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/bd8117ddceee465fa0129fc1fb32f8db-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/bd8117ddceee465fa0129fc1fb32f8db-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 30 COLUMNS
At line 3687 RHS
At line 3713 BOUNDS
At line 

In [21]:
total_score.item()

2352.0

In [22]:
scores

,team,gw_score
0,"[abdoulaye_doucoure, christian_walton, diogo_d...",37.0
1,"[bukayo_saka, chris_wood, ederson_santana_de_m...",56.0
2,"[antoine_semenyo, bukayo_saka, danny_welbeck, ...",96.0
3,"[antoine_semenyo, daniel_munoz, danny_welbeck,...",46.0
4,"[bryan_mbeumo, dwight_mcneil, eberechi_eze, er...",67.0
5,"[antoine_semenyo, bryan_mbeumo, david_raya_mar...",54.0
6,"[andrew_robertson, bryan_mbeumo, cole_palmer, ...",52.0
7,"[anthony_gordon, bernd_leno, brennan_johnson, ...",52.0
8,"[bryan_mbeumo, cole_palmer, danny_welbeck, dea...",88.0
9,"[antoine_semenyo, chris_wood, cole_palmer, dea...",58.0
